In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Schema definition & read csv
from pyspark.sql.types import *
    
# Create the schema for the table
orderSchema = StructType([
    StructField("deck", StringType()),
    StructField("maindeck", StringType()),
    StructField("count", IntegerType()),
    StructField("card", StringType())
    ])
    
# Import all files from bronze folder of lakehouse
df = spark.read.format("csv").option("header", "true").option("delimiter", ";").schema(orderSchema).load("Files/decks_bronze/*.csv")
    
# Display the first 10 rows of the dataframe to preview your data
display(df.head(10))

StatementMeta(, 7d684634-095f-4135-b12d-e56673818e49, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, afce93a4-a060-4319-9088-5053aa80d195)

In [3]:
# Define the schema for the sales_silver table
    
from pyspark.sql.types import *
from delta.tables import *
    
DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.decks_silver") \
    .addColumn("deck", StringType()) \
    .addColumn("card", StringType()) \
    .addColumn("maindeck", StringType()) \
    .addColumn("count", IntegerType()) \
    .addColumn("Created_or_Modified_TS", TimestampType()) \
    .execute()

StatementMeta(, 4450e732-4e81-4624-be47-637280d3e236, 5, Finished, Available, Finished, False)

In [2]:
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name
    
# Add columns IsFlagged, CreatedTS and ModifiedTS
df = df.withColumn("Created_or_Modified_TS", current_timestamp())
    


StatementMeta(, 33cd6c2f-af71-4190-a6df-98aa5c925aa0, 4, Finished, Available, Finished, False)

In [3]:
display(df.head(100))

StatementMeta(, 33cd6c2f-af71-4190-a6df-98aa5c925aa0, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 367551e6-ed8f-4992-87f6-bdc5d770f38a)

In [4]:
# Update existing records and insert new ones based on a condition defined by the columns SalesOrderNumber, OrderDate, CustomerName, and Item.

from delta.tables import *
    
deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/decks_silver')
    
dfUpdates = df
    
deltaTable.alias('silver') \
  .merge(
    dfUpdates.alias('updates'),
    'silver.deck = updates.deck and silver.card = updates.card and silver.maindeck = updates.maindeck'
  ) \
   .whenMatchedUpdate(set =
    {
      "maindeck": "updates.maindeck",
      "count": "updates.count",
      "Created_or_Modified_TS": "updates.Created_or_Modified_TS" 
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "deck": "updates.deck",
      "card": "updates.card",
      "maindeck": "updates.maindeck",
      "count": "updates.count",
      "Created_or_Modified_TS": "updates.Created_or_Modified_TS"
    }
  ) \
  .whenNotMatchedBySourceDelete() \
  .execute()

StatementMeta(, 33cd6c2f-af71-4190-a6df-98aa5c925aa0, 6, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.types import *
from delta.tables import *
    
# Create customer_gold dimension delta table
DeltaTable.createIfNotExists(spark) \
    .tableName("dbo.dimcard") \
    .addColumn("CardName", StringType()) \
    .addColumn("CardID", LongType()) \
    .execute()

StatementMeta(, 07b3473c-74d1-42aa-bf3d-29c550665cc3, 5, Finished, Available, Finished)

In [11]:
df = spark.read.table("dbo.decks_silver")


from pyspark.sql.functions import col, split
    
# Create customer_silver dataframe
    
dfdimCard_silver = df.dropDuplicates(["card"]).select(col("Card")) 
    
    
# Display the first 10 rows of the dataframe to preview your data

display(dfdimCard_silver.head(10))

StatementMeta(, 4450e732-4e81-4624-be47-637280d3e236, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a106c08e-df17-4993-a8b8-beb0ce9c9fae)

In [13]:
display(df.head(100))

StatementMeta(, 4450e732-4e81-4624-be47-637280d3e236, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 488bb6f5-022b-43a2-b9ac-aeda91b4e9b5)

In [3]:
from pyspark.sql.functions import monotonically_increasing_id, col, when, coalesce, max, lit
    
dfdimCard_temp = spark.read.table("dbo.dimCard")
    
MAXCardID = dfdimCard_temp.select(coalesce(max(col("CardID")),lit(0)).alias("MAXCardID")).first()[0]
    
dfdimCard_gold = dfdimCard_silver.join(dfdimCard_temp,(dfdimCard_silver.Card == dfdimCard_temp.CardName) , "left_anti")
    
dfdimCard_gold = dfdimCard_gold.withColumn("CardID",monotonically_increasing_id() + MAXCardID + 1)

# Display the first 10 rows of the dataframe to preview your data

display(dfdimCard_gold.head(10))

StatementMeta(, 76d09bd9-8b02-4415-b638-3a996be2fa15, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, aed04834-6ccb-4c17-a760-59ca3d42da03)

In [9]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 'Tables/dbo/dimcard')
    
dfUpdates = dfdimCard_gold
    
deltaTable.alias('gold') \
  .merge(
    dfUpdates.alias('updates'),
    'gold.CardName = updates.Card'
  ) \
   .whenMatchedUpdate(set =
    {
          
    }
  ) \
 .whenNotMatchedInsert(values =
    {
      "CardName": "updates.Card",
      "CardID": "updates.CardID"
    }
  ) \
  .execute()

StatementMeta(, 76d09bd9-8b02-4415-b638-3a996be2fa15, 11, Finished, Available, Finished)

In [1]:
# Load data to the dataframe as a starting point to create the gold layer
df = spark.read.table("dbo.decks_silver")

dfdimcard_temp = spark.read.table("dbo.dimCard")
display(dfdimcard_temp.head(10))
display(df.head(10))

StatementMeta(, 855f2f66-0076-4d50-bb05-9604acad2690, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1837b105-6310-44e3-a44a-73729489792e)

SynapseWidget(Synapse.DataFrame, f4342a6d-a03a-448d-8c89-cf4f49edd384)